# 🚀 Đồng Bộ Tự Động 30 Bộ Phim Tài Liệu Discovery (4K 60fps MAX BITRATE + SEO + QA) Thẳng Vào Thư Mục Google Drive (`1wZmfSYFBWDwhydvtFGR9eK-GKspjvJlz`)
Notebook này chạy 100% trên **Google Cloud (Colab)**, tự động kéo toàn bộ **30 video 4K 60fps (`3840x2160 @ 60fps`, âm thanh chuẩn `-14 LUFS`, đã khử logo AI & nâng nét Lanczos + CAS)** từ hệ thống Render Đa Luồng GitHub Cloud (`dung11122d/cloud-video-studio`) và đẩy thẳng vào đúng thư mục Google Drive `https://drive.google.com/drive/folders/1wZmfSYFBWDwhydvtFGR9eK-GKspjvJlz` qua Google Drive API v3 — **không tải bất kỳ byte nào về máy tính cá nhân**.

In [ ]:
import os
import re
import json
import urllib.request
from collections import defaultdict
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# 1. Xác thực trực tiếp với Google Drive API trên Cloud
auth.authenticate_user()
drive_service = build('drive', 'v3')

ROOT_FOLDER_ID = '1wZmfSYFBWDwhydvtFGR9eK-GKspjvJlz'
SUBFOLDER_NAME = 'Rendered_4K60_Discovery_Masters'

# Tạo hoặc lấy thư mục con Rendered_4K60_Discovery_Masters ngay bên trong folder 1wZmfSYFBWDwhydvtFGR9eK-GKspjvJlz
q = f"'{ROOT_FOLDER_ID}' in parents and name = '{SUBFOLDER_NAME}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
res = drive_service.files().list(q=q, fields='files(id, name)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
folders = res.get('files', [])
if folders:
    target_folder_id = folders[0]['id']
else:
    meta = {'name': SUBFOLDER_NAME, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [ROOT_FOLDER_ID]}
    target_folder_id = drive_service.files().create(body=meta, fields='id', supportsAllDrives=True).execute()['id']

print(f'📁 Đã kết nối thẳng vào thư mục Google Drive đích: https://drive.google.com/drive/folders/{target_folder_id}')

# Kiểm tra các file đã có sẵn trên Drive để tránh upload trùng
existing = {}
page_token = None
while True:
    r = drive_service.files().list(
        q=f"'{target_folder_id}' in parents and trashed = false",
        fields='nextPageToken, files(id, name, size)',
        pageToken=page_token,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()
    for f in r.get('files', []):
        existing[f['name']] = int(f.get('size', 0))
    page_token = r.get('nextPageToken')
    if not page_token:
        break

# 2. Lấy danh sách toàn bộ 30 video 4K 60fps + SEO JSON + QA Report từ GitHub Cloud Release
API_URL = 'https://api.github.com/repos/dung11122d/cloud-video-studio/releases/tags/discovery-4k60-masters'
req = urllib.request.Request(API_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as resp:
    rel = json.loads(resp.read().decode('utf-8'))
assets = rel.get('assets', [])

os.makedirs('/content/tmp_sync', exist_ok=True)

def upload_to_drive(local_path, filename):
    fsize = os.path.getsize(local_path)
    if filename in existing and existing[filename] == fsize:
        print(f'⏭️ Đã có trên Drive: {filename} ({fsize/(1024*1024):.1f} MB)')
        return
    mime = 'video/mp4' if filename.endswith('.mp4') else 'application/json'
    media = MediaFileUpload(local_path, mimetype=mime, chunksize=64*1024*1024, resumable=True)
    body = {'name': filename, 'parents': [target_folder_id]}
    request = drive_service.files().create(body=body, media_body=media, fields='id', supportsAllDrives=True)
    response = None
    while response is None:
        status, response = request.next_chunk()
    print(f'✅ Đã đẩy thẳng lên Google Drive: {filename} ({fsize/(1024*1024):.1f} MB)')

# Tách nhóm file đơn và file >2GB chia part (.part00, .part01...)
single_assets = [a for a in assets if not re.search(r'\.part\d+$', a['name'])]
part_groups = defaultdict(list)
for a in assets:
    m = re.match(r'^(.+\.mp4)\.part(\d+)$', a['name'])
    if m:
        part_groups[m.group(1)].append((int(m.group(2)), a))

print(f'🎬 Bắt đầu chuyển {len(single_assets)} file đơn và {len(part_groups)} video >2GB Max-Bitrate từ GitHub Cloud sang Google Drive...')

for idx, a in enumerate(single_assets, 1):
    name = a['name']
    if name in existing and existing[name] == a['size']:
        print(f'⏭️ [{idx}/{len(single_assets)}] Đã có sẵn trên Drive: {name}')
        continue
    tmp_file = f'/content/tmp_sync/{name}'
    print(f'⬇️ [{idx}/{len(single_assets)}] Kéo trên Cloud {name} ({a["size"]/(1024*1024):.1f} MB)...')
    urllib.request.urlretrieve(a['browser_download_url'], tmp_file)
    upload_to_drive(tmp_file, name)
    os.remove(tmp_file)

for base_name, parts in sorted(part_groups.items()):
    parts.sort(key=lambda x: x[0])
    total_sz = sum(p[1]['size'] for p in parts)
    if base_name in existing and existing[base_name] == total_sz:
        print(f'⏭️ Đã có sẵn bản Max-Bitrate trên Drive: {base_name} ({total_sz/(1024*1024):.1f} MB)')
        continue
    merged_path = f'/content/tmp_sync/{base_name}'
    print(f'🎬 Đang ghép nối {len(parts)} phần của siêu video 4K60 Max-Bitrate {base_name} ({total_sz/(1024*1024):.1f} MB) trên Cloud...')
    with open(merged_path, 'wb') as out_f:
        for p_idx, p_asset in parts:
            p_tmp = f'/content/tmp_sync/{p_asset["name"]}'
            urllib.request.urlretrieve(p_asset['browser_download_url'], p_tmp)
            with open(p_tmp, 'rb') as in_f:
                while True:
                    buf = in_f.read(64 * 1024 * 1024)
                    if not buf:
                        break
                    out_f.write(buf)
            os.remove(p_tmp)
    upload_to_drive(merged_path, base_name)
    os.remove(merged_path)

print(f'\n🎉 HOÀN TẤT 100%! Toàn bộ 30 Video 4K 60fps Master + SEO JSON đã nằm trong thư mục Google Drive của bạn: https://drive.google.com/drive/folders/{target_folder_id}')